In [1]:
import PyPDF2
import re
import numpy as np

In [101]:
def extract_archery_results(pdf_path):
    # Lire le PDF
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"

    #print("debug text : ",text)
    # Initialiser les données
    archers = []
    
    # Séparer le texte en lignes
    lines = text.split('\n')

    # Variables pour stocker les informations de l'archer actuel
    current_archer = None

    in_scores = False

    for line in lines:
        line = line.strip()

        # Détecter le nom de l'archer
        if line.startswith("Archer:"):
            current_archer = {
                "Nom": line.replace("Archer:", "").strip(),
                "Id_Club": None,
                "Club": None,
                "Catégorie": None,
                "Scores": []
            }
            if current_archer:
                archers.append(current_archer)
                #print("debug current_archer :",current_archer)
            continue

        # Détecter le club et la catégorie
     
        if line.startswith("Clubs / Pays:"):
            if current_archer:
                parts = line.replace("Clubs / Pays:", "").strip().split()
                # Extraire le club (tout ce qui précède la catégorie)
                club_parts = []
                for part in parts:
                    if part.isdigit() or part == "-":
                        club_parts.append(part)
                    else:
                        break
                id_club = " ".join(club_parts).strip()
                id_club = ''.join(re.sub('[-]', '', id_club).split())
                current_archer["Id_Club"] = id_club
                # La catégorie est le reste de la ligne
                nom_club = " ".join(parts[len(club_parts):]).strip()
                cible = re.findall(r'[1-9][ABCD]|[1-9][0-9][ABCD]', nom_club)
                nom_club = re.sub(cible[0], '', nom_club)
                current_archer["Club"] = nom_club
            #print("debug Id_Club :",current_archer["Id_Club"])
            #print("debug Club :",current_archer["Club"])   
            continue

        # Détecter la catégorie (CO S1H, etc.)
        if line.startswith("CO"):
            current_archer["Catégorie"] = line#.strip()
            #print("debug Catégorie CO :",current_archer["Catégorie"])
            continue

        # Détecter la catégorie (CL S1H, etc.)
        if line.startswith("CL"):
            current_archer["Catégorie"] = line#.strip()
            #print("debug Catégorie CL :",current_archer["Catégorie"])
            continue

        # Détecter les scores après "Somme Tot. 10+XX"
        if "Somme Tot. 10+XX" in line:
            in_scores = True
            score_lines = []
            continue

        # Extraire les 12 lignes de scores
        if in_scores:
            score_lines.append(line)
            if len(score_lines) == 12:
                # Traiter les 12 lignes de scores
                score = ['','','','','','']
                for i in range(0, 12, 2):
                    # Ligne impaire : numéro de volée + 3 scores
                    impair_line = ''.join(score_lines[i].split())
                    id_volee = impair_line[0]
                    score[0:3] = re.findall(r'10|[1-9XM]', impair_line[1:-1])[:3] 

                    # Ligne paire : 3 scores
                    pair_line = ''.join(score_lines[i + 1].split())
                    score[3:-1] = re.findall(r'10|[1-9XM]', pair_line[0:-1])[:3] 

                    # Ajouter les scores pour cette volée
                    current_archer["Scores"].append({
                        "Volée": id_volee,
                        "Score1": score[0],
                        "Score2": score[1],
                        "Score3": score[2],
                        "Score4": score[3],
                        "Score5": score[4],
                        "Score6": score[5]
                    })
                in_scores = False
            continue


    # Ajouter le dernier archer
    if current_archer:
        archers.append(current_archer)

    return archers

In [102]:
# Exemple d'utilisation
pdf_path = "/home/cperrot/Téléchargements/Feuilles de marque Qualification FCL & HCO.pdf"
results = extract_archery_results(pdf_path)

# Afficher les résultats
for archer in results:
    print(f"Nom: {archer['Nom']}")
    print(f"Club/Pays: {archer['Id_Club']}")
    print(f"Club/Pays: {archer['Club']}")
    print(f"Catégorie: {archer['Catégorie']}")
    for score in archer["Scores"]:
        print(f"  {score['Volée']}: {score['Score1']} + {score['Score2']} + {score['Score3']} + {score['Score4']} + {score['Score5']} + {score['Score6']}")
    print("---")

Nom: Piron Benjamin
Club/Pays: 0895293
Club/Pays: Sarcelles
Catégorie: CO S1H
  1: X + 10 + 9 + 9 + 8 + 8
  2: X + X + 10 + 9 + 8 + 7
  3: 10 + 10 + 8 + 8 + 8 + 7
  4: X + X + 10 + 9 + 9 + 6
  5: X + X + 10 + 9 + 9 + 8
  6: X + 10 + 10 + 10 + 9 + 9
---
Nom: Piron Benjamin
Club/Pays: 0895293
Club/Pays: Sarcelles
Catégorie: CO S1H
  1: X + 10 + 9 + 9 + 9 + 9
  2: X + X + 10 + 10 + 10 + 10
  3: X + X + 9 + 8 + 8 + 8
  4: 10 + 10 + 9 + 9 + 9 + 9
  5: X + 10 + 10 + 10 + 9 + 9
  6: X + X + 10 + 9 + 9 + 8
---
Nom: Cadronet Nathan
Club/Pays: 0892199
Club/Pays: Suresnes
Catégorie: CO S1H
  1: X + X + 10 + 10 + 9 + 9
  2: X + 10 + 10 + 9 + 9 + 9
  3: X + X + 10 + 9 + 9 + 9
  4: X + 10 + 10 + 10 + 9 + 8
  5: X + 10 + 10 + 10 + 10 + 8
  6: X + X + 10 + 10 + 10 + 10
---
Nom: Cadronet Nathan
Club/Pays: 0892199
Club/Pays: Suresnes
Catégorie: CO S1H
  1: X + X + X + X + 10 + 9
  2: 10 + 10 + 9 + 9 + 9 + 8
  3: X + 10 + 10 + 10 + 10 + 9
  4: 10 + 10 + 10 + 10 + 9 + 9
  5: X + X + X + 10 + 10 + 9
  6: X

In [56]:
aa = "X10 929 21"
bb = ''.join(aa.split())
print(aa)
print(bb)


cc = re.findall(r'10|[1-9XM]', bb)[:3]  # ["10", "X", "9"]
print(cc)

X10 929 21
X1092921
['X', '10', '9']


In [87]:
aa = "Saint Denis De La Reunion7A"

cc = re.findall(r'[1-9][ABCD]|[1-9][0-9][ABCD]', aa)
print(cc)

bb = re.sub(cc[0], '', aa)
print(bb)

['7A']
Saint Denis De La Reunion
